<a href="https://colab.research.google.com/github/Ali-Shahrez/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [27]:
%cd /content
!rm -rf flyrank-ml-internship
!git clone -q https://github.com/Ali-Shahrez/flyrank-ml-internship.git
%cd flyrank-ml-internship

import sys, os
import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows")

sys.path.append(os.path.abspath('scripts'))
from ml_utils import precision_at_k, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

RANDOM_STATE = 42  # fixed for reproducibility — same seed used throughout w02-w04

/content
/content/flyrank-ml-internship
Loaded 30,000 rows


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task shape:** Lane 2 is a ranking task ("which pages should a reviewer look at first"), not plain classification — established back in w02. Per `training-honest-models`, the fix for ranking tasks is not to predict a yes/no label directly, but to train a classifier on the observed label (`is_declining_label`) and use its **predicted probability** as the ranking score, evaluated with Precision@K. That keeps the same Precision@20/50 metric as the Week 4 baseline, so the comparison is apples-to-apples.

**Method ladder — readable, then stronger:**

1. **Logistic Regression first.** It's the simplest classifier that outputs a probability: each feature gets one learned coefficient (sign + magnitude both readable), which matches the "readable → stronger" principle used for the depth-2-then-depth-4 decision tree comparison in w02. It's also a genuine step up from the Week 4 rule — the rule combined exactly two hand-picked features (`days_since_last_update`, `impressions_90d`) with a fixed formula; Logistic Regression *learns* weights across 15+ candidate features instead of guessing the formula.
2. **Random Forest next, if the comparison earns it.** Per the skill: "add complexity only when the comparison earns it." Section 3 shows Logistic Regression barely clearing the test base rate, which is the trigger to try a tree ensemble that can capture feature interactions a linear model can't.

**Why not just re-tune the Week 4 rule instead of training a model?** Before building anything, we checked whether the rule's own choice of feature (`impressions_90d`) was simply the wrong pick — testing whether `ctr` would have done better. It didn't: `ctr` correlates with `is_declining_label` at -0.103 (train) / -0.129 (test), similar in size and direction to `impressions_90d`'s -0.073 / -0.172. Several individual, hand-picked features skew in the "wrong" direction relative to the label on their own — which is the actual argument for a model that weighs many features jointly, rather than betting on one or two by hand the way the rule does. Fixing the rule using something we learned from inspecting the test set would also mean tuning the baseline to the split it's being judged on — the same mistake the Week 2 hand-rule made, just in reverse — so the rule was left untouched and the model was built to actually beat it honestly.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-holdout split, using the exact canonical method from `scripts/03_train_model.py`** (not an approximation): unique `client_id`s are shuffled with `np.random.default_rng(42)`, and the top `round(n_clients * 0.20)` clients become the test set — 6 of 32 clients here. This guarantees no client's pages appear in both train and test, which matters because pages from the same client can share patterns a model could memorize rather than genuinely learn from (the same reasoning that made the Week 2 hand-rule collapse on held-out clients).

This reproduces the documented canonical split exactly: **27,675 train rows / 2,325 test rows / 39.1% test base rate** on the full 30k population.

**Eligibility gate, applied on top of the split, not instead of it:** both sides are then filtered to `impressions_90d >= 500` — the same floor Week 4 used, justified by the same reason (CTR and other rate metrics are unreliable below a minimum volume). Filtering *after* splitting, rather than splitting the already-filtered population, keeps the split methodology identical to earlier notebooks.

**A real limitation surfaced by this design, not hidden from it:** after the eligibility filter, the test side shrinks to only **618 rows across 4 of the 6 test clients**, with one client (`client_f74efabef1`) contributing 598 of those 618 rows (96.8%). This is a genuine consequence of an honest client-level split on a 30k-row dataset — small enough that any single Precision@K number from this test set should be read with real caution, not treated as precise to the decimal. This instability is discussed further in Section 4.

In [28]:
# --- Canonical client-holdout split, exact reference method (scripts/03_train_model.py) ---
client_series = df['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
train_df = df[~test_mask].copy()
test_df  = df[test_mask].copy()

print(f"Unique clients: {len(unique_clients)}  |  test clients: {test_client_count}")
print(f"Train rows: {len(train_df):,}  |  Test rows: {len(test_df):,}")
print(f"Test base rate (declining share): {(test_df['trend_direction']=='down').mean():.3f}")
# Expect: Train 27,675 | Test 2,325 | base rate 0.391 — matches the documented canonical split

Unique clients: 32  |  test clients: 6
Train rows: 27,675  |  Test rows: 2,325
Test base rate (declining share): 0.391


In [29]:
# --- Apply the eligibility gate (same floor as Week 4) on top of the canonical split ---
MIN_IMPRESSIONS = 500

train_eligible = train_df[train_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()
test_eligible  = test_df[test_df['impressions_90d'] >= MIN_IMPRESSIONS].copy()

for frame in (train_eligible, test_eligible):
    frame['is_declining_label'] = (frame['trend_direction'] == 'down').astype(int)

print(f"Train eligible: {len(train_eligible):,} rows | base rate {train_eligible['is_declining_label'].mean():.3f}")
print(f"Test eligible:  {len(test_eligible):,} rows | base rate {test_eligible['is_declining_label'].mean():.3f}")
print(f"\nTest client counts:\n{test_eligible['client_id'].value_counts()}")

Train eligible: 16,108 rows | base rate 0.598
Test eligible:  618 rows | base rate 0.519

Test client counts:
client_id
client_f74efabef1    598
client_d4735e3a26      9
client_4fc82b26ae      7
client_0b918943df      4
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Recomputing the baseline on the same test split (not quoting the Week 4 whole-population number).** Week 4's official numbers (P@20=0.800, P@50=0.560) were computed on the entire eligible population (16,726 rows, no split — a fixed rule doesn't need one). That number stays correct and unchanged in the Week 4 notebook. But comparing a model's *test-split* score against a *whole-population* baseline number would not be an honest same-split comparison, so the decline-blind rule is recomputed here, using the identical formula, on `test_eligible` only.

In [30]:
# --- Recompute the Week 4 decline-blind baseline on the TEST-eligible slice only ---
# Same formula as w04's staleness_visibility_score_v2 — no trend_direction/trend_pct anywhere in it.
test_eligible['baseline_score'] = (
    test_eligible['days_since_last_update'].rank(pct=True) +
    test_eligible['impressions_90d'].rank(pct=True)
)

baseline_p20_test = precision_at_k(test_eligible['is_declining_label'], test_eligible['baseline_score'], k=20)
baseline_p50_test = precision_at_k(test_eligible['is_declining_label'], test_eligible['baseline_score'], k=50)

print(f"Baseline (test-split eligible, n={len(test_eligible):,}):")
print(f"  Precision@20: {baseline_p20_test:.3f}")
print(f"  Precision@50: {baseline_p50_test:.3f}")
print(f"\nFor reference — Week 4 baseline, full eligible population (n=16,726): P@20=0.800, P@50=0.560")

Baseline (test-split eligible, n=618):
  Precision@20: 0.250
  Precision@50: 0.240

For reference — Week 4 baseline, full eligible population (n=16,726): P@20=0.800, P@50=0.560


**Finding — the baseline collapses on this test split, and it's below its own base rate (0.519).** Two compounding causes, checked directly rather than assumed:

1. **A different tie-mass dominates this split.** 88% of test-eligible rows (543/618) share `days_since_last_update == 20` — a different synthetic-data artifact from the `==104` mass that dominated the full population in Week 4. With that term nearly constant, the rank-sum score collapses to ranking by `impressions_90d` alone.
2. **On this split, `impressions_90d` is negatively correlated with the label** (test: -0.172, train: -0.073) — the highest-impression pages are somewhat *less* likely to be labeled declining (declining share by impressions quartile, test: 0.548 / 0.552 / 0.584 / **0.394** for Q1→Q4). So a score that effectively ranks by impressions alone points the wrong way on this split.

This is a real result, not a bug — a rule hand-tuned to look good on one population doesn't automatically transfer to another, especially with a 4-client, 618-row test slice. The rule was **not** modified after finding this, to avoid tuning the baseline to the split it's being judged against.

### Feature matrix

Missingness in `word_count`/`char_count`/`search_volume`/`competition`/`cpc` tracks `content_type` (confirmed below: `feedly article` rows are 100% missing keyword-context columns; `keyword article` rows are ~30% missing `word_count`/`char_count`). A blind `fillna(0)` — what the reference pipeline does — would silently encode content type as a fake zero, per the `flyrank-data` skill's explicit warning. `has_*` flags are added before filling, so real missingness stays visible to the model as its own signal.

In [31]:
# Confirm the missingness-follows-content_type pattern before building around it
missingness_check_cols = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc']
print("Missingness (%) by content_type — train_eligible:")
print((train_eligible.groupby('content_type')[missingness_check_cols]
       .apply(lambda g: g.isna().mean() * 100)).round(1))

Missingness (%) by content_type — train_eligible:
                    word_count  char_count  search_volume  competition    cpc
content_type                                                                 
comparison article         0.0         0.0            0.0          0.0    0.0
feedly article             0.0         0.0          100.0        100.0  100.0
keyword article           29.8        29.8            0.8          0.8    0.8


In [32]:
MISSINGNESS_TRACKED_COLS = ['word_count', 'char_count', 'search_volume', 'competition', 'cpc']

def add_derived_and_flags(frame):
    frame = frame.copy()
    # has_* flags BEFORE filling — captures real missingness, not a fake zero
    for col in MISSINGNESS_TRACKED_COLS:
        frame[f'has_{col}'] = frame[col].notna().astype(int)
    # log1p transforms (heavy-tailed traffic counts), matching MODEL_NUMERIC_FEATURES
    frame['log_impressions_90d'] = np.log1p(frame['impressions_90d'])
    frame['log_clicks_90d']      = np.log1p(frame['clicks_90d'])
    frame['log_sessions_90d']    = np.log1p(frame['sessions_90d'])
    frame['log_ai_sessions_90d'] = np.log1p(frame['ai_sessions_90d'])
    return frame

train_eligible = add_derived_and_flags(train_eligible)
test_eligible  = add_derived_and_flags(test_eligible)

def build_features(frame, numeric_cols, categorical_cols, flag_cols):
    numeric = frame[numeric_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    flags = frame[flag_cols]
    categorical = frame[categorical_cols].fillna('unknown').astype(str)
    dummies = pd.get_dummies(categorical, prefix=categorical_cols, dtype=float)
    return pd.concat([numeric.reset_index(drop=True), flags.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

numeric_cols = [c for c in MODEL_NUMERIC_FEATURES if c in train_eligible.columns]
categorical_cols = [c for c in MODEL_CATEGORICAL_FEATURES if c in train_eligible.columns]
flag_cols = [f'has_{c}' for c in MISSINGNESS_TRACKED_COLS]

X_train = build_features(train_eligible, numeric_cols, categorical_cols, flag_cols)
X_test  = build_features(test_eligible,  numeric_cols, categorical_cols, flag_cols)

# Align test columns to train's — fit-on-train-only discipline applied to feature structure, not just values
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_eligible['is_declining_label']
y_test  = test_eligible['is_declining_label']

print(f"X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}")
assert not any(c in X_train.columns for c in ['trend_direction', 'trend_pct', 'content_id', 'client_id']), \
    "Leakage check failed — label-source or ID column present in features"
print("Leakage check passed: no trend_direction / trend_pct / content_id / client_id in the feature matrix.")

X_train shape: (16108, 56)  |  X_test shape: (618, 56)
Leakage check passed: no trend_direction / trend_pct / content_id / client_id in the feature matrix.


In [33]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled  = scaler.transform(X_test)         # test only ever gets transformed, never fit

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_scaled, y_train)

# Probability of "declining" (class 1) IS the ranking score
logreg_test_scores = logreg.predict_proba(X_test_scaled)[:, 1]

logreg_p20 = precision_at_k(y_test, logreg_test_scores, k=20)
logreg_p50 = precision_at_k(y_test, logreg_test_scores, k=50)

print(f"Logistic Regression — Precision@20: {logreg_p20:.3f}")
print(f"Logistic Regression — Precision@50: {logreg_p50:.3f}")

Logistic Regression — Precision@20: 0.550
Logistic Regression — Precision@50: 0.480


In [34]:
from sklearn.ensemble import RandomForestClassifier

# Hyperparameters match scripts/03_train_model.py's random_forest config, for consistency with
# the established reference methodology
rf = RandomForestClassifier(
    class_weight="balanced_subsample",
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf.fit(X_train, y_train)  # trees don't need scaled features

rf_test_scores = rf.predict_proba(X_test)[:, 1]

rf_p20 = precision_at_k(y_test, rf_test_scores, k=20)
rf_p50 = precision_at_k(y_test, rf_test_scores, k=50)

print(f"Random Forest — Precision@20: {rf_p20:.3f}")
print(f"Random Forest — Precision@50: {rf_p50:.3f}")

Random Forest — Precision@20: 0.850
Random Forest — Precision@50: 0.860


In [35]:
# --- THE comparison table: baseline vs model(s), same split, same metrics, base rate included ---
comparison_table = pd.DataFrame({
    'Method': ['Baseline (decline-blind rule)', 'Logistic Regression', 'Random Forest', 'Test base rate'],
    'Precision@20': [baseline_p20_test, logreg_p20, rf_p20, y_test.mean()],
    'Precision@50': [baseline_p50_test, logreg_p50, rf_p50, y_test.mean()],
})
print(f"All rows evaluated on the identical test-eligible split, n={len(y_test)}:\n")
print(comparison_table.to_string(index=False))

All rows evaluated on the identical test-eligible split, n=618:

                       Method  Precision@20  Precision@50
Baseline (decline-blind rule)      0.250000      0.240000
          Logistic Regression      0.550000      0.480000
                Random Forest      0.850000      0.860000
               Test base rate      0.519417      0.519417


**Reading the table honestly, against the base rate (0.519) — not just against the broken baseline:**

| Method | P@20 | P@50 | Beats base rate? |
|---|---:|---:|---|
| Baseline (decline-blind rule) | 0.250 | 0.240 | No — well below chance |
| Logistic Regression | 0.550 | 0.480 | Barely / no — P@50 below chance |
| Random Forest | 0.850 | 0.860 | Yes, clearly |

Both models beat the broken baseline, but beating a broken baseline isn't the same as beating chance. Logistic Regression clears P@20 by only 0.031 over the 0.519 base rate and *loses* to chance at P@50 — a genuinely weak result, reported as-is rather than dressed up. Random Forest is the only method that clearly and substantially beats both the baseline and the base rate at both K values. Whether that 0.86 number can be trusted at that precision is a separate question, addressed in Section 4.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [36]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 15 Random Forest feature importances:")
print(importances.head(15))

Top 15 Random Forest feature importances:
content_age_days          0.107091
avg_position              0.083071
log_clicks_90d            0.077021
scroll_rate               0.076976
ctr                       0.064977
days_with_sessions        0.061195
char_count                0.045326
word_count                0.044474
age_tier_365+             0.042532
log_sessions_90d          0.041578
log_impressions_90d       0.038429
days_since_last_update    0.026173
age_tier_91-180           0.026122
days_with_impressions     0.025972
engagement_rate           0.024747
dtype: float64


**What the model leans on — plausible, not suspicious.** No single feature dominates (top is `content_age_days` at 0.107); the next several are `avg_position`, `log_clicks_90d`, `scroll_rate`, `ctr`, `days_with_sessions` — all directly plausible drivers of decline (older content, worse ranking position, weaker click/scroll engagement). Nothing named `trend_direction`, `trend_pct`, `content_id`, or `client_id` appears (also confirmed programmatically in Section 3's leakage check). Per the skill's own test — "does the top feature make sense, or is it suspiciously perfect?" — this passes.

In [37]:
# Sanity-check: is the 0.86 test score a sign of overfitting? Compare against TRAIN performance,
# and check whether the win is secretly riding on one client rather than genuine signal.
train_scores_rf = rf.predict_proba(X_train)[:, 1]
train_p20_rf = precision_at_k(y_train, train_scores_rf, k=20)
train_p50_rf = precision_at_k(y_train, train_scores_rf, k=50)
print(f"Random Forest — TRAIN Precision@20: {train_p20_rf:.3f}, Precision@50: {train_p50_rf:.3f}")

top50 = pd.Series(rf_test_scores, index=y_test.index).sort_values(ascending=False).head(50)
print(f"\nOf the top 50 ranked test pages, {y_test.loc[top50.index].sum()} of 50 are actually declining.")
print(f"By chance at this test's base rate, you'd expect ~{50*y_test.mean():.1f}.")

top50_clients = test_eligible.loc[top50.index, 'client_id'].value_counts()
print(f"\nClient concentration in top-50 predictions:\n{top50_clients}")
print(f"\nFor context, test_eligible client counts overall:\n{test_eligible['client_id'].value_counts()}")

Random Forest — TRAIN Precision@20: 1.000, Precision@50: 1.000

Of the top 50 ranked test pages, 43 of 50 are actually declining.
By chance at this test's base rate, you'd expect ~26.0.

Client concentration in top-50 predictions:
client_id
client_f74efabef1    50
Name: count, dtype: int64

For context, test_eligible client counts overall:
client_id
client_f74efabef1    598
client_d4735e3a26      9
client_4fc82b26ae      7
client_0b918943df      4
Name: count, dtype: int64


**Train Precision@20/50 = 1.000 — a real overfitting warning sign,** confirmed by a `min_samples_leaf` sweep below rather than assumed from one number. The top-50 test predictions come entirely from `client_f74efabef1` — at first glance this looked like the model detecting one client rather than genuine decline signal (the same collapse pattern the baseline showed), but checking the underlying counts corrected that read: `client_f74efabef1` makes up 598 of 618 test-eligible rows (96.8%) — there simply isn't enough of the other three clients (4, 7, and 9 rows) to appear in a top-50 list at all. That part is a consequence of the split's client imbalance, not evidence of client-identity leakage on its own.

In [38]:
# Regularization sweep — does test performance track train's, or is the 1.000 train score independent noise?
sweep_rows = []
for leaf_size in [25, 50, 100, 200]:
    rf_variant = RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=leaf_size,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
    )
    rf_variant.fit(X_train, y_train)
    train_p50_v = precision_at_k(y_train, rf_variant.predict_proba(X_train)[:, 1], k=50)
    test_p20_v  = precision_at_k(y_test,  rf_variant.predict_proba(X_test)[:, 1],  k=20)
    test_p50_v  = precision_at_k(y_test,  rf_variant.predict_proba(X_test)[:, 1],  k=50)
    sweep_rows.append((leaf_size, train_p50_v, test_p20_v, test_p50_v))
    print(f"min_samples_leaf={leaf_size:>4}  |  train P@50={train_p50_v:.3f}  |  test P@20={test_p20_v:.3f}  test P@50={test_p50_v:.3f}")

min_samples_leaf=  25  |  train P@50=1.000  |  test P@20=0.850  test P@50=0.860
min_samples_leaf=  50  |  train P@50=1.000  |  test P@20=0.800  test P@50=0.780
min_samples_leaf= 100  |  train P@50=0.980  |  test P@20=0.650  test P@50=0.740
min_samples_leaf= 200  |  train P@50=0.980  |  test P@20=0.750  test P@50=0.660


**Interpretation of the sweep — and the honest limit of this result.** Train P@50 barely moves (1.000 → 1.000 → 0.980 → 0.980) even as tree complexity is pulled back hard, while test P@20/P@50 swing noisily (0.85/0.86 → 0.80/0.78 → 0.65/0.74 → 0.75/0.66) with no clean trend. That's not the pattern you'd expect from resolving overfitting through regularization — it's the signature of a test set too small (618 rows, effectively 1 dominant client) to pin down a precise Precision@K at all.

**What's still true, and what isn't:**
- **Robust finding:** at *every* `min_samples_leaf` setting tested, Random Forest clears both the recomputed baseline (0.250/0.240) and the test base rate (0.519) by a wide margin. The direction of the result — a learned model beats a hand-tuned rule and beats chance — holds up.
- **Not trustworthy:** the specific value 0.86. Given the volatility above, this test split cannot support claiming a precise number to two decimal places. The honest range to report is something like "0.65–0.86 depending on tree regularization, all clearly above the 0.519 base rate" — not a single confident 0.86.
- **Three concrete wrong cases**, from the client not represented in the top-50 (small test clients, where the model had almost no training signal to learn from): the 4-row `client_0b918943df` (75% actually declining) and 7-row `client_4fc82b26ae` (14.3% declining) are both too thin for the model — or any method — to say anything reliable about; the model effectively can't be evaluated on them separately, which is itself a limitation worth naming rather than ignoring.

**Bottom line:** this result should be read as "Random Forest beats the baseline and chance, directionally and repeatably" — a real, useful finding for the capstone — not as "Random Forest achieves 86% precision," which overstates what a 618-row, single-client-dominated test set can actually support. The capstone's planned move to the full warehouse (with enough clients and rows for a stable holdout, and a genuine future-window label instead of the same-window proxy used here) is the direct fix for this limitation, not a nice-to-have.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.